# Urdu-Seg: toolkit baselines and runtime

Run this notebook top to bottom (Jupyter or Google Colab, Python 3.9–3.12) from anywhere inside the cloned repository.
It tokenizes the clean and the noisy corpus with spaCy, NLTK, Stanza and Indic NLP, times every tokenizer including
Urdu-Seg, and scores everything against the gold standard. Prediction files are written to `results/clean/` and
`results/noisy/`; afterwards `python reproduce.py` recomputes all tables.

Urdu-Seg and whitespace predictions must already exist in `results/` (they are shipped, or run `python reproduce.py`).
Runtimes depend on your hardware; the paper's numbers were measured on the machine described in Section 5.3.

In [ ]:
# 1) Find the repository folder (works wherever the notebook is opened) and make it the working folder
import os, sys, glob, zipfile
def _find_root():
    here = os.getcwd()
    starts = [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here)), os.path.expanduser("~")]
    for s in starts:
        cands = [s] + glob.glob(os.path.join(s, "*")) + glob.glob(os.path.join(s, "*", "*")) + glob.glob(os.path.join(s, "*", "*", "*"))
        for c in cands:
            if os.path.isfile(os.path.join(c, "src", "urduseg.py")):
                return os.path.abspath(c)
    raise FileNotFoundError("Could not find the repository folder (the one that contains src/urduseg.py).")
ROOT = _find_root()
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))
print("working in", ROOT)

In [ ]:
# 2) Install baseline libraries (takes a few minutes)
%pip install "spacy>=3.7,<3.9" nltk stanza psutil
%pip install indic-nlp-library

In [ ]:
# 3) Download models / data
import types
try:
    import nltk
    for pkg in ("punkt", "punkt_tab"):
        try: nltk.download(pkg, quiet=True)
        except Exception as e: print("nltk", pkg, e)
except Exception as e:
    nltk = types.SimpleNamespace(__version__="not installed"); print("NLTK not available:", e)
try:
    import stanza
    stanza.download("ur", processors="tokenize", verbose=False)
except Exception as e:
    stanza = types.SimpleNamespace(__version__="not installed"); print("Stanza not available:", e)

In [ ]:
# 4) Tokenizers (a baseline that is not installed is skipped, the rest still run)
import sys, csv, json, time, statistics, platform, types
os.chdir(ROOT); sys.path.insert(0, os.path.join(ROOT, "src"))   # ROOT comes from cell 1
csv.field_size_limit(min(sys.maxsize, 2**31 - 1))
import spacy
from urduseg import load_default, make_spacy_tokenizer

seg = load_default()                                   # Urdu-Seg with the shipped lexicon + gazetteer
nlp_seg = spacy.blank("ur"); nlp_seg.tokenizer = make_spacy_tokenizer(nlp_seg, seg)
nlp_default = spacy.blank("ur")                        # spaCy's built-in Urdu tokenizer

TOKENIZERS = {
    "Urdu-Seg":         seg.tokenize,
    "Urdu-Seg (spaCy)": lambda t: [x.text for x in nlp_seg(t)],
    "Whitespace":       str.split,
    "spaCy":            lambda t: [x.text for x in nlp_default(t)],
}
missing = []
try:
    from nltk.tokenize import word_tokenize, wordpunct_tokenize
    word_tokenize("test")
    TOKENIZERS["NLTK-word"] = lambda t: word_tokenize(t)
    TOKENIZERS["NLTK-wordpunct"] = wordpunct_tokenize
except Exception as e:
    missing.append(f"NLTK ({type(e).__name__}: {str(e)[:80]})")
try:
    stz = stanza.Pipeline("ur", processors="tokenize", use_gpu=False, verbose=False)
    TOKENIZERS["Stanza"] = lambda t: [tok.text for s in stz(t).sentences for tok in s.tokens]
except Exception as e:
    missing.append(f"Stanza ({type(e).__name__}: {str(e)[:80]})")
try:
    from indicnlp.tokenize import indic_tokenize
    TOKENIZERS["IndicNLP"] = lambda t: indic_tokenize.trivial_tokenize(t, lang="ur")
except Exception as e:
    missing.append(f"IndicNLP ({type(e).__name__}: {str(e)[:80]})")
try:                                                    # UrduHack needs old TensorFlow; often fails
    from urduhack.tokenization import word_tokenizer as _uh
    TOKENIZERS["UrduHack"] = _uh
except Exception as e:
    missing.append(f"UrduHack ({type(e).__name__})")
if missing:
    print("Skipped (not installed or failed to load): " + "; ".join(missing))

sample = "ہم تو ٹی وی دیکھنے «کے» عادی نہیں ہیں۔ ❤️ کردیا"
for k, f in TOKENIZERS.items():
    print(f"{k:18}", " | ".join(f(sample)))

In [ ]:
# 5) Tokenize the clean and noisy corpora and write prediction files
def read_corpus(path):
    with open(path, encoding="utf-8-sig") as fh:
        rows = list(csv.DictReader(fh))
    col = "preprocessed" if "preprocessed" in rows[0] else "raw"
    return [(r["id"], r[col]) for r in rows]

CORPORA = {"clean": "data/inputs/preprocessed_5000.csv", "noisy": "data/inputs/noisy_raw_5000.csv"}
errors = {}
for version, path in CORPORA.items():
    rows = read_corpus(path)
    os.makedirs(f"results/{version}", exist_ok=True)
    for name, f in TOKENIZERS.items():
        if name in ("Urdu-Seg", "Whitespace"):
            continue                                    # already produced by predict.py
        out, fails = [], 0
        for sid, text in rows:
            try:
                toks = f(text)
            except Exception:
                toks, fails = text.split(), fails + 1
            out.append((sid, toks))
        errors[(version, name)] = fails
        safe = name.replace(" ", "").replace("(", "-").replace(")", "")
        with open(f"results/{version}/baseline_{safe}.csv", "w", encoding="utf-8-sig", newline="") as fh:
            w = csv.writer(fh); w.writerow(["id", "tokens"])
            for sid, toks in out:
                w.writerow([sid, json.dumps(toks, ensure_ascii=False)])
        print(version, name, "done", f"({fails} sentences raised errors)" if fails else "")

In [ ]:
# 6) Runtime: full clean corpus, 1 warm-up + 5 timed runs per tokenizer
rows = [t for _, t in read_corpus(CORPORA["clean"])]
timing = {}
for name, f in TOKENIZERS.items():
    f(rows[0])
    runs, ntok = [], 0
    for _ in range(5):
        t0 = time.perf_counter()
        ntok = sum(len(f(t)) for t in rows)
        runs.append(time.perf_counter() - t0)
    m = statistics.mean(runs)
    timing[name] = {"mean_s": m, "sd_s": statistics.stdev(runs), "sentences": len(rows), "tokens": ntok,
                    "ms_per_sentence": 1000 * m / len(rows), "sentences_per_s": len(rows) / m,
                    "tokens_per_s": ntok / m}
    print(f"{name:18} {m:9.4f} s ± {statistics.stdev(runs):.4f}   {len(rows)/m:12,.0f} sent/s")
env = {"python": platform.python_version(), "platform": platform.platform(), "processor": platform.processor(),
       "spacy": spacy.__version__, "nltk": nltk.__version__, "stanza": stanza.__version__}
try:
    import psutil; env["ram_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
except Exception: pass
json.dump({"timing": timing, "environment": env, "tokenizer_errors": {f"{k[0]}/{k[1]}": v for k, v in errors.items()}},
          open("results/timing.json", "w", encoding="utf-8"), indent=2)
print(env)

In [ ]:
# 7) Score everything on the held-out test split (clean and noisy); works on Windows, macOS, Linux, Colab
import glob, subprocess
SETS = (("clean", "independent", "data/v3/v2dedup_revised.csv"),   # primary: double-annotated, deduplicated
        ("clean", "full", "data/v3/gold_v3.csv"),                 # secondary: all unique test comments
        ("noisy", "full", "data/v3/gold_v3_noisy.csv"))
for version, setname, gold in SETS:
    cmd = [sys.executable, "src/evaluate.py", "--gold", gold, "--split", "test", "--ref", "Whitespace",
           "--out", f"results/metrics_{version}_{setname}_test_all.json",
           "--pred", f"Urdu-Seg=results/{version}/pred_urduseg.csv",
           "--pred", f"Whitespace=results/{version}/pred_whitespace.csv"]
    for f in sorted(glob.glob(f"results/{version}/baseline_*.csv")):
        cmd += ["--pred", os.path.basename(f)[len("baseline_"):-4] + "=" + f]
    print(f"== {version} input, {setname} test set")
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")   # Windows: read output as UTF-8
    res = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", env=env)
    print(res.stdout or res.stderr)